# Risk Factor Analysis and Patient Profiling for Early Stroke Detection

Exploratory data analysis on the Kaggle **Stroke Prediction Dataset** (5,110 patients).
The goal is to identify which patient attributes separate stroke cases from non-cases well enough to
justify a proactive screening program, and to establish a clean, reproducible baseline for predictive
modeling.

**Author:** Sultan Abdul Fatah · **Dataset:** [`fedesoriano/stroke-prediction-dataset`](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset) · **Environment:** Python 3.10+, Jupyter

| Stage | Output |
|---|---|
| 1. Setup & Data Loading | Raw data via `kagglehub` (5,110 x 12) |
| 2. Data Profiling | Missingness, duplicates, anomalies, outliers, class balance |
| 3. Data Cleaning | Documented cleaning log (5,109 x 12) |
| 4. Exploratory Analysis | Four questions, rate-based comparisons, confounding checks |
| 5. Findings & Next Steps | Insights translated into screening rules |

> The dataset is downloaded automatically, so no local CSV is required to run the notebook.

In [ ]:
import os
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')

# Dataset is pulled on every run; nothing is stored in the repository.
path = kagglehub.dataset_download("fedesoriano/stroke-prediction-dataset")
csv_files = sorted(f for f in os.listdir(path) if f.endswith('.csv'))
if not csv_files:
    raise FileNotFoundError(f"No CSV file found in {path}")

df_raw = pd.read_csv(os.path.join(path, csv_files[0]))
df = df_raw.copy()
print(f"Loaded: {csv_files[0]}")
print(f"Shape (rows, columns): {df.shape}")

The raw dataset holds **5,110 rows and 12 columns**. One row is one patient, one column per attribute.
`stroke` is the binary target (1 = patient had a stroke).

## 1. Data Understanding

| Group | Columns |
|---|---|
| Identifier | `id` |
| Demographics | `gender`, `age`, `ever_married`, `work_type`, `Residence_type` |
| Medical history | `hypertension`, `heart_disease`, `avg_glucose_level`, `bmi`, `smoking_status` |
| Target | `stroke` |

In [ ]:
display(df.head())

print("=" * 100)
print("Dataset Information:\n")
df.info()

## 2. Data Profiling

Every claim below is backed by a code check: missingness, duplicates, category cardinality,
cross-column consistency, outlier ranges, and target balance.

In [ ]:
# --- Missing values ---
print("Missing Values\n")
missing_df = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Percentage (%)': (df.isnull().mean() * 100).round(2)
})
display(missing_df[missing_df['Missing Count'] > 0])

print("\n" + "=" * 50 + "\n")

# --- Duplicates ---
print("Duplicates\n")
print(f"Full row duplicates: {df.duplicated().sum()}")
print(f"Duplicate IDs      : {df['id'].duplicated().sum()}")

print("\n" + "=" * 50 + "\n")

# --- Category cardinality and spelling consistency ---
print("Categorical Value Counts\n")
cat_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
for col in cat_cols:
    print(f"{col}:\n{df[col].value_counts(dropna=False)}\n")

print("=" * 50 + "\n")

# --- Cross-column consistency and impossible values ---
print("Consistency & Anomaly Checks\n")
print(f"Age < 18 AND ever_married == 'Yes' : {len(df[(df['age'] < 18) & (df['ever_married'] == 'Yes')])}")
print(f"Age < 10 AND smoker/former smoker  : {len(df[(df['age'] < 10) & (df['smoking_status'].isin(['smokes', 'formerly smoked']))])}")
children_age = df.loc[df['work_type'] == 'children', 'age']
print(f"work_type 'children' age range     : {children_age.min()} - {children_age.max()}")
print(f"Infants (age < 1 year)             : {len(df[df['age'] < 1])}")
print(f"Extreme BMI (> 60)                 : {len(df[df['bmi'] > 60])}")

print("\n" + "=" * 50 + "\n")

# --- Numeric distribution and IQR outliers ---
print("Numeric Distribution & IQR Outliers\n")
num_cols = ['age', 'avg_glucose_level', 'bmi']
display(df[num_cols].describe().T[['min', '25%', '50%', '75%', 'max', 'mean', 'std']])

for col in num_cols:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((df[col] < lower) | (df[col] > upper)).sum())
    print(f"{col}: skew={df[col].skew():.2f} | IQR outliers={n_out} ({n_out / len(df) * 100:.2f}%) | bounds=[{lower:.2f}, {upper:.2f}]")

print("\n" + "=" * 50 + "\n")

# --- Target balance ---
print("Target Distribution\n")
target_df = pd.DataFrame({
    'Count': df['stroke'].value_counts(),
    'Percentage (%)': (df['stroke'].value_counts(normalize=True) * 100).round(2)
})
display(target_df)

print("\n" + "=" * 50 + "\n")

# --- Is BMI missingness informative? ---
print("Stroke rate (%) by BMI missingness\n")
display((pd.crosstab(df['bmi'].isnull(), df['stroke'], normalize='index') * 100).round(2))

## 3. Data Cleaning

Cleaning is driven by the profiling evidence above. The raw frame is never overwritten, and the log
below records what changed, how many rows it touched, and why.

| # | Column | Problem | Action | Rationale |
|---|---|---|---|---|
| 1 | `id` | Unique identifier, no clinical meaning | Drop column | Carries no signal; would only add noise to a model |
| 2 | `gender` | 1 row with `Other` (0.02%) | Drop row | Too small a group to analyse; not a stroke case |
| 3 | `bmi` | 201 missing (3.93%) | Add flag `bmi_is_missing`, then impute gender-group median (Female 27.8 / Male 28.4) | Missing-BMI patients show a 19.90% stroke rate vs 4.26% — dropping them would erase signal |
| 4 | `bmi` | 13 values above 60 (max 97.6) | Cap at 60.0 | BMI near 100 is clinically implausible, likely a data-entry error |
| 5 | `avg_glucose_level` | 627 IQR outliers (max 271.74) | Keep | Hyperglycaemia is real and is itself a stroke risk factor |
| 6 | `smoking_status` | 1,544 `Unknown` (30.22%) | Keep as its own category | Too large to drop; forced imputation would bias the lifestyle analysis |

**Rows before cleaning:** 5,110 → **after:** 5,109. All 249 stroke cases preserved.

In [ ]:
df_clean = df_raw.copy()
print(f"Shape before cleaning : {df_clean.shape}")

# 1. Identifier column carries no analytical value
df_clean = df_clean.drop(columns=['id'])

# 2. Single 'Other' gender row: too small to analyse, not a stroke case
df_clean = df_clean[df_clean['gender'] != 'Other'].copy()

# 3. Missing BMI is informative -> keep that signal as an explicit flag, then impute
#    with the median per gender (robust to the right-skewed distribution)
df_clean['bmi_is_missing'] = df_clean['bmi'].isnull().astype(int)
df_clean['bmi'] = df_clean['bmi'].fillna(
    df_clean.groupby('gender')['bmi'].transform('median')
)

# 4. Cap implausible BMI values (13 rows > 60, max 97.6) instead of dropping rows
df_clean['bmi'] = df_clean['bmi'].clip(upper=60.0)

# 5 and 6. Glucose outliers and 'Unknown' smoking status are kept deliberately (see cleaning log)

print(f"Shape after cleaning  : {df_clean.shape}")
print(f"Remaining missing     : {int(df_clean.isnull().sum().sum())}")
print(f"Stroke cases preserved: {int(df_clean['stroke'].sum())} of {int(df_raw['stroke'].sum())}")
print()
display(df_clean.head())

In [ ]:
# Persist the cleaned frame to the notebook working directory (git-ignored).
df_clean.to_csv('stroke_data_clean.csv', index=False)
print(f"Clean dataset written: stroke_data_clean.csv | shape={df_clean.shape}")

## 4. Exploratory Data Analysis

Four questions, written before any chart. Each comparison uses **stroke rate (%)** instead of raw
counts so groups of different sizes stay comparable, and every chart carries the overall baseline
(**4.87%**) as a reference line.

| # | Question | Method |
|---|---|---|
| 1 | How does stroke rate differ across age groups? | Age binning + bar chart vs baseline |
| 2 | How large is the effect of hypertension, heart disease, and both combined? | Four-group comorbidity comparison |
| 3 | How do glucose level and BMI relate to stroke? | Glucose bands (bar) + BMI distribution (boxplot) |
| 4 | Does smoking status matter, or is age confounding it? | Smoking rate (bar) + age distribution (boxplot) |

In [ ]:
df_eda = df_clean.copy()
overall_rate = df_eda['stroke'].mean() * 100
print(f"Baseline stroke rate: {overall_rate:.2f}%  ({int(df_eda['stroke'].sum())}/{len(df_eda)})")

### Q1 — Age groups

**Hypothesis:** stroke risk rises with age, with the sharpest jump after 60.

In [ ]:
df_eda['age_group'] = pd.cut(
    df_eda['age'],
    bins=[0, 30, 45, 60, 75, 100],
    labels=['<30', '30-44', '45-59', '60-74', '75+'],
    right=False
)

age_stats = df_eda.groupby('age_group', observed=False)['stroke'].agg(['count', 'sum', 'mean']).reset_index()
age_stats['rate'] = age_stats['mean'] * 100

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(age_stats['age_group'].astype(str), age_stats['rate'], color=sns.color_palette('Reds', len(age_stats)))
ax.axhline(overall_rate, color='black', linestyle='--', linewidth=1.2,
           label=f'Overall baseline ({overall_rate:.2f}%)')

for i, row in age_stats.iterrows():
    ax.text(i, row['rate'] + 0.5, f"{row['rate']:.2f}%\n({int(row['sum'])}/{int(row['count'])})",
            ha='center', fontsize=9)

ax.set_title('Stroke Rate (%) by Age Group', fontweight='bold')
ax.set_xlabel('Age Group (years)')
ax.set_ylabel('Stroke Rate (%)')
ax.set_ylim(0, 24)
ax.legend()
plt.tight_layout()
plt.show()

**Observation:** the rate climbs from 0.13% (<30) and 0.79% (30-44) to 4.83% (45-59), 9.21% (60-74),
peaking at 19.69% (75+).

**Interpretation:** age is the strongest single discriminator in this dataset — the 75+ group sits at
roughly 4x the overall baseline.

**Insight:** treat 45 as the entry point for preventive education and 60+ (especially 75+) as the
priority cohort for routine screening.

### Q2 — Hypertension and heart disease (cumulative effect)

**Hypothesis:** each comorbidity raises the stroke rate, and both together raise it further.

In [ ]:
def comorbidity_label(row):
    h, hd = row['hypertension'], row['heart_disease']
    if h == 0 and hd == 0:
        return 'Neither'
    if h == 1 and hd == 0:
        return 'Hypertension only'
    if h == 0 and hd == 1:
        return 'Heart disease only'
    return 'Hypertension + heart disease'

df_eda['comorbidity'] = df_eda.apply(comorbidity_label, axis=1)
order_com = ['Neither', 'Hypertension only', 'Heart disease only', 'Hypertension + heart disease']

com_stats = df_eda.groupby('comorbidity')['stroke'].agg(['count', 'sum', 'mean']).reindex(order_com).reset_index()
com_stats['rate'] = com_stats['mean'] * 100

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.bar(com_stats['comorbidity'], com_stats['rate'], color=sns.color_palette('OrRd', len(com_stats)))
ax.axhline(overall_rate, color='black', linestyle='--', linewidth=1.2,
           label=f'Overall baseline ({overall_rate:.2f}%)')

for i, row in com_stats.iterrows():
    ax.text(i, row['rate'] + 0.5, f"{row['rate']:.2f}%\n({int(row['sum'])}/{int(row['count'])})",
            ha='center', fontsize=9, bbox=dict(facecolor='white', edgecolor='none', pad=1, alpha=0.8))

ax.set_title('Stroke Rate (%) by Hypertension & Heart Disease History', fontweight='bold')
ax.set_xlabel('Medical History Combination')
ax.set_ylabel('Stroke Rate (%)')
ax.set_ylim(0, 25)
ax.legend()
plt.tight_layout()
plt.show()

**Observation:** 3.39% (neither) → 12.21% (hypertension only) → 16.04% (heart disease only) →
20.31% (both).

**Interpretation:** the effect is cumulative. Having both comorbidities puts a patient at roughly 6x
the rate of a patient with neither.

**Insight:** an automatic early-warning flag in the EHR for patients with either diagnosis — and
especially both — lets the hospital schedule preventive follow-up before symptoms appear.

### Q3 — Glucose level and BMI

**Hypothesis:** high glucose separates stroke cases; BMI on its own does not.

In [ ]:
df_eda['glucose_cat'] = pd.cut(
    df_eda['avg_glucose_level'],
    bins=[0, 100, 126, 200, 300],
    labels=['Normal\n(<100)', 'Borderline\n(100-125)', 'High\n(126-199)', 'Very high\n(>=200)'],
    right=False
)
gluc_stats = df_eda.groupby('glucose_cat', observed=False)['stroke'].agg(['count', 'sum', 'mean']).reset_index()
gluc_stats['rate'] = gluc_stats['mean'] * 100

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5))

axes[0].bar(gluc_stats['glucose_cat'].astype(str), gluc_stats['rate'], color=sns.color_palette('YlOrRd', len(gluc_stats)))
axes[0].axhline(overall_rate, color='black', linestyle='--', linewidth=1.2,
                label=f'Baseline ({overall_rate:.2f}%)')
for i, row in gluc_stats.iterrows():
    axes[0].text(i, row['rate'] + 0.4, f"{row['rate']:.2f}%\n({int(row['sum'])}/{int(row['count'])})",
                 ha='center', fontsize=8.5)
axes[0].set_title('Stroke Rate (%) by Average Glucose Band', fontweight='bold')
axes[0].set_xlabel('Average Glucose (mg/dL)')
axes[0].set_ylabel('Stroke Rate (%)')
axes[0].set_ylim(0, 16)
axes[0].legend()

sns.boxplot(data=df_eda, x='stroke', y='bmi', color='#5c85b8', width=0.5, ax=axes[1])
axes[1].set_title('BMI Distribution: Non-Stroke (0) vs Stroke (1)', fontweight='bold')
axes[1].set_xlabel('Stroke (0 = No, 1 = Yes)')
axes[1].set_ylabel('BMI')

plt.tight_layout()
plt.show()

**Observation:** 3.58% (normal) and 3.71% (borderline) sit near the baseline, then 8.06% (126-199) and
12.90% (>=200). The BMI boxplots overlap almost completely, with both medians near 28.

**Interpretation:** glucose above 126 mg/dL tracks stroke risk closely, reaching 3.6x the normal band,
while BMI alone does not separate the two classes.

**Insight:** routine glucose screening is a better triage input than BMI. Patients at or above
126 mg/dL belong in a risk-management program even when their BMI looks normal.

### Q4 — Smoking status, with age as a confounder

**Hypothesis:** the smoking signal is mostly explained by age.

In [ ]:
smoke_order = ['formerly smoked', 'smokes', 'never smoked', 'Unknown']
smoke_stats = df_eda.groupby('smoking_status')['stroke'].agg(['count', 'sum', 'mean']).reindex(smoke_order).reset_index()
smoke_stats['rate'] = smoke_stats['mean'] * 100

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5))

axes[0].bar(smoke_stats['smoking_status'], smoke_stats['rate'], color=sns.color_palette('Blues_r', len(smoke_stats)))
axes[0].axhline(overall_rate, color='black', linestyle='--', linewidth=1.2,
                label=f'Baseline ({overall_rate:.2f}%)')
for i, row in smoke_stats.iterrows():
    axes[0].text(i, row['rate'] + 0.25, f"{row['rate']:.2f}%\n({int(row['sum'])}/{int(row['count'])})",
                 ha='center', fontsize=8.5)
axes[0].set_title('Stroke Rate (%) by Smoking Status', fontweight='bold')
axes[0].set_xlabel('Smoking Status')
axes[0].set_ylabel('Stroke Rate (%)')
axes[0].set_ylim(0, 10)
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend()

sns.boxplot(data=df_eda, x='smoking_status', y='age', order=smoke_order, color='#5c85b8', width=0.5, ax=axes[1])
axes[1].set_title('Age Distribution by Smoking Status (Confounding Check)', fontweight='bold')
axes[1].set_xlabel('Smoking Status')
axes[1].set_ylabel('Age (years)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

# Same comparison restricted to adults 45+, where the age structure is closer
adults = df_eda[df_eda['age'] >= 45]
print("Stroke rate (%) by smoking status, age >= 45:")
print(
    adults.groupby('smoking_status')['stroke'].agg(['count', 'mean'])
    .assign(rate_pct=lambda d: (d['mean'] * 100).round(2))
    .drop(columns='mean')
    .reindex(smoke_order)
)

**Observation:** `formerly smoked` 7.92% > `smokes` 5.32% > `never smoked` 4.76% > `Unknown` 3.04%.
Median age: `formerly smoked` ~57, `smokes` and `never smoked` ~47, `Unknown` ~23.

**Interpretation:** this is not evidence that quitting smoking causes stroke. Age is the confounder —
the `Unknown` group is dominated by children and young adults, while `formerly smoked` holds the
oldest patients.

**Insight:** lifestyle risk must always be read alongside age, and smoking history for adults should be
recorded more completely to shrink the 1,544-row `Unknown` group.

## 5. Findings & Next Steps

| Finding | Metric | Implication |
|---|---|---|
| Age dominates | 0.13% (<30) → 19.69% (75+), ~4x baseline | Screen from 45, prioritise 60+ |
| Comorbidities compound | 3.39% → 20.31%, ~6x | Auto-flag hypertensive and cardiac patients |
| Glucose beats BMI | 12.90% at >=200 mg/dL vs 3.58% normal | Prioritise glucose screening over BMI |
| Smoking signal is age-confounded | 7.92% vs 3.04%, median age 57 vs 23 | Always control for age; improve data capture |
| Missing BMI is informative | 19.90% vs 4.26% | Keep `bmi_is_missing` as a feature; never drop rows |

**Next steps — from EDA to predictive modeling**

1. **Baseline models:** logistic regression for interpretable coefficients, gradient boosting
   (XGBoost/LightGBM) as the performance ceiling.
2. **Class imbalance (249 vs 4,861):** class weights, SMOTE applied inside cross-validation folds
   only, and threshold tuning on the precision-recall curve.
3. **Metrics:** ROC-AUC, PR-AUC, and recall at a fixed precision. Not accuracy, which a
   majority-class baseline already scores at 95%.
4. **Validation:** stratified k-fold plus a held-out test set that is never resampled.
5. **Feature work:** test `bmi_is_missing` as a standalone predictor, encode smoking status as
   ordinal, add age x comorbidity interaction terms.
6. **Interpretation:** SHAP values to confirm the drivers found in this EDA and to explain individual
   risk scores to clinicians.

---

**Author:** Sultan Abdul Fatah · GitHub [@Sultvnnnn](https://github.com/Sultvnnnn) · Dataset: [Kaggle Stroke Prediction Dataset](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset)